## RQ1 

"""
RQ1: Coverage / Residual analysis
Phase 2.8 — comparing taxonomy (6 labels) vs LLooM-induced (8 labels) coverage
on score_sample_wide.parquet (53,283 comments)

In [1]:

import pandas as pd
 
# ---- Load ----
wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")
 
# EDIT these to match your actual column names in score_sample_wide.parquet

# 6 hand-written SSBC taxonomy labels (title-case columns)
# 6 hand-written SSBC taxonomy labels (title-case columns)
LLOOM_COLS = [
    "Workplace Problem Guidance",
    "Career Planning Advice",
    "Emotional Support",
    "Critical Pushback",
    "Personal Relating",
    "Discussion Direction",
    "Practical Advice",
    "Support Connections"
]
 
# 8 frozen LLooM-induced concepts
TAXONOMY_COLS = [
    # "Practical Advice",
    # "Support Connections",
    "informational_support",
    "emotional_support",
    "esteem_support",
    "tangible_support",
    "network_support",
    "unsupportive_response",
]
# ---- Diagnostic: confirm columns exist before computing anything ----
missing_tax = [c for c in TAXONOMY_COLS if c not in wide.columns]
missing_lloom = [c for c in LLOOM_COLS if c not in wide.columns]
if missing_tax or missing_lloom:
    print("MISSING COLUMNS — fix before proceeding:")
    print("  taxonomy missing:", missing_tax)
    print("  lloom missing:   ", missing_lloom)
    print("\nActual columns in wide:", list(wide.columns))
    raise SystemExit
 
# ---- Core RQ1 numbers ----
tax_hit = wide[TAXONOMY_COLS].sum(axis=1) > 0
lloom_hit = wide[LLOOM_COLS].sum(axis=1) > 0
 
n_total = len(wide)
taxonomy_coverage = tax_hit.mean()
induced_coverage = lloom_hit.mean()
residual = (lloom_hit & ~tax_hit).mean()
both = (lloom_hit & tax_hit).mean()
neither = (~lloom_hit & ~tax_hit).mean()
 
print(f"N comments (unweighted):     {n_total:,}")
print(f"Taxonomy coverage:           {taxonomy_coverage:.4f}")
print(f"Induced (LLooM) coverage:    {induced_coverage:.4f}")
print(f"Residual (LLooM-only):       {residual:.4f}")
print(f"Both taxonomy AND LLooM:     {both:.4f}")
print(f"Neither (pct_none):          {neither:.4f}")

N comments (unweighted):     53,283
Taxonomy coverage:           0.8947
Induced (LLooM) coverage:    0.9745
Residual (LLooM-only):       0.0979
Both taxonomy AND LLooM:     0.8765
Neither (pct_none):          0.0074


In [2]:

# ---- Reweighted version (accounting for sysadmin oversample, w ≈ 10.73) ----
if "w" in wide.columns:
    def weighted_mean(mask, w):
        return (mask * w).sum() / w.sum()
 
    print("\n--- Reweighted (w column) ---")
    print(f"Taxonomy coverage (w):    {weighted_mean(tax_hit, wide['w']):.4f}")
    print(f"Induced coverage (w):     {weighted_mean(lloom_hit, wide['w']):.4f}")
    print(f"Residual (w):             {weighted_mean(lloom_hit & ~tax_hit, wide['w']):.4f}")
else:
    print("\nNo 'w' column found — skipping reweighted version.")
 
# ---- Per-label coverage (solo + any) for both halves ----
rows = []
for col in TAXONOMY_COLS + LLOOM_COLS:
    family = "taxonomy" if col in TAXONOMY_COLS else "lloom"
    rows.append({
        "label": col,
        "family": family,
        "total_yes": int(wide[col].sum()),
        "pct_of_comments": wide[col].mean(),
    })
per_label = pd.DataFrame(rows).sort_values("total_yes", ascending=False)
print("\n--- Per-label breakdown ---")
print(per_label.to_string(index=False))
 
# ---- Save outputs ----
out = pd.DataFrame([{
    "n_total": n_total,
    "taxonomy_coverage": taxonomy_coverage,
    "induced_coverage": induced_coverage,
    "residual": residual,
    "both": both,
    "neither": neither,
}])
out.to_csv("rq1_coverage_summary.csv", index=False)
per_label.to_csv("rq1_per_label_coverage.csv", index=False)
print("\nSaved: rq1_coverage_summary.csv, rq1_per_label_coverage.csv")


--- Reweighted (w column) ---
Taxonomy coverage (w):    0.8722
Induced coverage (w):     0.9738
Residual (w):             0.1192

--- Per-label breakdown ---
                     label   family  total_yes  pct_of_comments
     informational_support taxonomy      41502         0.778912
          Practical Advice    lloom      30745         0.577122
         Personal Relating    lloom      27124         0.509065
      Discussion Direction    lloom      24415         0.458308
         Critical Pushback    lloom      24064         0.451719
    Career Planning Advice    lloom      21347         0.400642
         Emotional Support    lloom      20031         0.375943
       Support Connections    lloom      18549         0.348129
         emotional_support taxonomy      16837         0.315998
Workplace Problem Guidance    lloom      15064         0.282722
            esteem_support taxonomy      14524         0.272639
           network_support taxonomy      12766         0.239593
         

In [3]:
# taxonomy coverage excluding informational_support
tax_no_infosupport = [c for c in TAXONOMY_COLS if c != "informational_support"]
print(wide[tax_no_infosupport].sum(axis=1).gt(0).mean())

0.7021939455360997


### findings- informational_support alone accounts for 77.9% of comments — it's carrying most of the taxonomy's coverage.

## RQ2 - response type × BAT construct

In [4]:
"""
RQ2 — Step 1: Diagnostics before running the mismatch test
Goal: join score_sample_wide.parquet (comment-level) to bat_posts_results_final_patched.csv
(post-level BAT constructs: EX, EMO, COG, MD) and figure out how to define a single
"dominant construct" per post before testing construct x response-type.

Run this first. Paste output back before we write the actual stats script.
"""

import pandas as pd

# ---- Load ----
wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")
bat = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/bat_posts_results_final_patched.csv")  # adjust path if needed

CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]  # adjust if actual column names differ

print("bat_posts columns:", list(bat.columns))
print("wide columns (relevant):", [c for c in wide.columns if c in ("id", "post_id")])

# ---- Sanity check construct columns exist ----
missing = [c for c in CONSTRUCT_COLS if c not in bat.columns]
if missing:
    print(f"\nMISSING CONSTRUCT COLUMNS: {missing}")
    print("Actual bat columns:", list(bat.columns))
    raise SystemExit

# ---- Convert YES/NO strings to bool ----
for c in CONSTRUCT_COLS:
    print(f"\n{c} unique raw values:", bat[c].unique())
    bat[c] = bat[c].astype(str).str.strip().str.upper().eq("YES")

# ---- Diagnostic: how many constructs flagged per post ----
n_flagged = bat[CONSTRUCT_COLS].sum(axis=1)
print("\n--- Constructs flagged per post ---")
print(n_flagged.value_counts().sort_index())
print(f"\n0 constructs:  {(n_flagged == 0).mean():.4f}")
print(f"1 construct:   {(n_flagged == 1).mean():.4f}")
print(f"2+ constructs: {(n_flagged >= 2).mean():.4f}")

# ---- Diagnostic: which construct combos are common among multi-flagged posts ----
multi = bat[n_flagged >= 2][CONSTRUCT_COLS]
if len(multi) > 0:
    combo_counts = multi.apply(lambda row: "+".join([c for c in CONSTRUCT_COLS if row[c]]), axis=1)
    print("\n--- Top construct combinations (posts with 2+ flags) ---")
    print(combo_counts.value_counts().head(10))

# ---- Diagnostic: MD-specific numbers (key construct for the mismatch hypothesis) ----
print(f"\nMD flagged (any):        {bat['MD'].mean():.4f}")
print(f"MD flagged alone only:   {((bat['MD']) & (n_flagged == 1)).mean():.4f}")

# ---- Key-match check: how many wide comments have a matching post_id in bat ----
bat_id_col = "post_id" if "post_id" in bat.columns else "id"
print(f"\nUsing bat key column: {bat_id_col}")

wide_ids = set(wide["post_id"].unique()) if "post_id" in wide.columns else None
bat_ids = set(bat[bat_id_col].unique())

if wide_ids is not None:
    match_rate = len(wide_ids & bat_ids) / len(wide_ids)
    print(f"Match rate (wide post_id found in bat): {match_rate:.4f}")
    print(f"Wide unique post_ids: {len(wide_ids):,}")
    print(f"Bat unique post_ids:  {len(bat_ids):,}")
    print(f"Unmatched wide post_ids: {len(wide_ids - bat_ids):,}")
else:
    print("wide has no post_id column — check schema")

bat_posts columns: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning']
wide columns (relevant): ['id', 'post_id']

EX unique raw values: ['NO' 'YES']

EMO unique raw values: ['NO' 'YES']

COG unique raw values: ['NO' 'YES']

MD unique raw values: ['NO' 'YES']

--- Constructs flagged per post ---
0    132415
1      7232
2      3043
3      1573
4       389
Name: count, dtype: int64

0 constructs:  0.9154
1 construct:   0.0500
2+ constructs: 0.0346

--- Top construct combinations (posts with 2+ flags) ---
EX+EMO           1410
EX+EMO+MD        1095
EMO+MD            640
EX+EMO+COG        391
EX+EMO+COG+MD     389
EX+COG            362
EX+MD             354
EMO+COG           247
EMO+COG+MD         45
EX+COG+MD          42
Name: count, dtype: int64

MD flagged (any):        0.0235
MD flagged alone only:   0.0055

Using bat key column: post_id
Match rate

In [6]:
"""
RQ2 — Step 2: Construct prevalence within the 7,998 posts that actually have comments
in score_sample_wide.parquet. This tells us whether we have enough MD-flagged posts
(the key construct for the mismatch hypothesis) to power the test.
"""

import pandas as pd

# ---- Load ----
wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")
bat = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/bat_posts_results_final_patched.csv")  # adjust path if needed

CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]
for c in CONSTRUCT_COLS:
    bat[c] = bat[c].astype(str).str.strip().str.upper().eq("YES")

# Restrict bat to posts that appear in the comment sample
sample_post_ids = wide["post_id"].unique()
bat_sample = bat[bat["post_id"].isin(sample_post_ids)].drop_duplicates(subset="post_id")

print(f"Posts in comment sample: {len(sample_post_ids):,}")
print(f"Matched rows in bat (deduped by post_id): {len(bat_sample):,}")

n_flagged = bat_sample[CONSTRUCT_COLS].sum(axis=1)
print("\n--- Constructs flagged per post (comment-sample posts only) ---")
print(n_flagged.value_counts().sort_index())

print("\n--- Per-construct prevalence (comment-sample posts only) ---")
for c in CONSTRUCT_COLS:
    n_yes = bat_sample[c].sum()
    print(f"{c}: {n_yes:,} posts flagged ({bat_sample[c].mean():.4f})")

# How many COMMENTS (not posts) map to each construct being flagged?
# This is the actual n for the stats test.
merged = wide.merge(bat_sample[["post_id"] + CONSTRUCT_COLS], on="post_id", how="left")
print("\n--- Per-construct comment counts (this is your test's n) ---")
for c in CONSTRUCT_COLS:
    n_comments = merged[c].sum()
    print(f"{c}: {n_comments:,} comments on flagged posts")

print(f"\nComments with NO construct flagged on parent post: {(merged[CONSTRUCT_COLS].sum(axis=1) == 0).sum():,}")

merged.to_parquet("rq2_merged_wide_bat.parquet")
print("\nSaved: rq2_merged_wide_bat.parquet")

Posts in comment sample: 7,998
Matched rows in bat (deduped by post_id): 7,998

--- Constructs flagged per post (comment-sample posts only) ---
1    4595
2    2036
3    1096
4     271
Name: count, dtype: int64

--- Per-construct prevalence (comment-sample posts only) ---
EX: 4,020 posts flagged (0.5026)
EMO: 5,137 posts flagged (0.6423)
COG: 1,433 posts flagged (0.1792)
MD: 2,449 posts flagged (0.3062)

--- Per-construct comment counts (this is your test's n) ---
EX: 27,856 comments on flagged posts
EMO: 35,468 comments on flagged posts
COG: 8,467 comments on flagged posts
MD: 19,949 comments on flagged posts

Comments with NO construct flagged on parent post: 0

Saved: rq2_merged_wide_bat.parquet


In [8]:
! pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 20.6 MB/s eta 0:00:00 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


###for each (construct, label) pair, it's asking: among comments on posts where this construct is flagged, is this response label more or less common than among comments where the construct isn't flagged?

In [10]:
"""
RQ2 — Step 3: The optimal-matching test
For each of 4 BAT constructs (EX, EMO, COG, MD) x 14 response-type labels:
  - weighted 2x2 contingency table (construct present/absent) x (label present/absent)
  - Cramer's V (= phi for 2x2) as effect size
  - standardized residuals to see direction (over/under-represented)
  - chi-square p-value, Holm-corrected across all 56 tests

Uses rq2_merged_wide_bat.parquet saved by step 2.
"""

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

merged = pd.read_parquet("rq2_merged_wide_bat.parquet")

CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]

TAXONOMY_COLS = [
    "informational_support", "emotional_support", "esteem_support",
    "tangible_support", "network_support", "unsupportive_response",
]
LLOOM_COLS = [
    "Workplace Problem Guidance", "Career Planning Advice", "Emotional Support",
    "Critical Pushback", "Personal Relating", "Discussion Direction",
    "Practical Advice", "Support Connections",
]
ALL_LABELS = TAXONOMY_COLS + LLOOM_COLS

w = merged["w"] if "w" in merged.columns else pd.Series(1.0, index=merged.index)

results = []
for construct in CONSTRUCT_COLS:
    c_flag = merged[construct].astype(bool)
    for label in ALL_LABELS:
        l_flag = merged[label].astype(bool)

        # weighted 2x2 table
        tab = np.zeros((2, 2))
        tab[0, 0] = w[(c_flag) & (l_flag)].sum()      # construct+, label+
        tab[0, 1] = w[(c_flag) & (~l_flag)].sum()     # construct+, label-
        tab[1, 0] = w[(~c_flag) & (l_flag)].sum()     # construct-, label+
        tab[1, 1] = w[(~c_flag) & (~l_flag)].sum()    # construct-, label-

        n = tab.sum()
        chi2, p, dof, expected = chi2_contingency(tab)
        cramers_v = np.sqrt(chi2 / n)  # phi for 2x2

        rate_in = tab[0, 0] / (tab[0, 0] + tab[0, 1])   # P(label | construct present)
        rate_out = tab[1, 0] / (tab[1, 0] + tab[1, 1])  # P(label | construct absent)

        # standardized residual for the construct+/label+ cell
        std_resid = (tab[0, 0] - expected[0, 0]) / np.sqrt(expected[0, 0])

        family = "taxonomy" if label in TAXONOMY_COLS else "lloom"

        results.append({
            "construct": construct,
            "label": label,
            "family": family,
            "rate_in_construct": rate_in,
            "rate_outside_construct": rate_out,
            "diff": rate_in - rate_out,
            "cramers_v": cramers_v,
            "std_resid_construct_label": std_resid,
            "chi2": chi2,
            "p_raw": p,
        })

res = pd.DataFrame(results)

# Holm correction across ALL 56 tests
reject, p_holm, _, _ = multipletests(res["p_raw"], method="holm")
res["p_holm"] = p_holm
res["sig_holm"] = reject

res = res.sort_values(["construct", "cramers_v"], ascending=[True, False])

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)
print(res[["construct", "label", "family", "rate_in_construct", "rate_outside_construct",
           "diff", "cramers_v", "p_holm", "sig_holm"]].to_string(index=False))

res.to_csv("rq2_construct_label_tests.csv", index=False)
print("\nSaved: rq2_construct_label_tests.csv")

# Quick summary: significant results only, sorted by effect size
sig = res[res["sig_holm"]].sort_values("cramers_v", ascending=False)
print(f"\n{len(sig)} of 56 tests significant after Holm correction")
print("\n--- Significant results, largest effect first ---")
print(sig[["construct", "label", "diff", "cramers_v", "p_holm"]].to_string(index=False))

construct                      label   family  rate_in_construct  rate_outside_construct      diff  cramers_v        p_holm  sig_holm
      COG          emotional_support taxonomy           0.429459                0.265797  0.163662   0.123783  0.000000e+00      True
      COG          Emotional Support    lloom           0.492171                0.329763  0.162408   0.116481  0.000000e+00      True
      COG        Support Connections    lloom           0.458218                0.318150  0.140068   0.101476  0.000000e+00      True
      COG           Practical Advice    lloom           0.680452                0.533496  0.146956   0.101243  0.000000e+00      True
      COG             esteem_support taxonomy           0.350440                0.240633  0.109807   0.086222  0.000000e+00      True
      COG           tangible_support taxonomy           0.296813                0.197354  0.099459   0.083504  0.000000e+00      True
      COG      informational_support taxonomy           0.8238

### Results interpretation: 
construct = COG, label = emotional_support, family = taxonomy
rate_in_construct = 0.429459
Take every comment sitting on a COG-flagged post. 42.9% of those comments got tagged emotional_support.

rate_outside_construct = 0.265797
Take every comment sitting on a post without COG flagged (i.e. EX, EMO, or MD posts, or combinations, but not COG). Only 26.6% of those comments got tagged emotional_support.

diff = 0.163662
The gap between those two rates: 0.429459 − 0.265797 = 0.1637, i.e. comments respond with emotional support ~16.4 percentage points more often when the parent post shows cognitive impairment than when it doesn't. This is the plain-English headline number.
cramers_v = 0.123783
The effect size — how strong the association is, on a 0-to-1 scale, independent of sample size. 0.12 is on the small-to-moderate side (rule of thumb for 2x2 tables: ~0.1 = small, ~0.3 = medium, ~0.5 = large). So the effect is real but not huge — COG explains some but not most of the variation in whether a comment gets tagged emotional_support.

### https://www.scirp.org/reference/referencespapers?referenceid=3845582

threshold depends on degress of freedom;??? need to check the threshold 

In [13]:
"""
RQ2 — Sort by effect size (Cramer's V), not just significance.
With n in the tens of thousands, almost everything is "significant" after Holm,
so Cramer's V is what actually tells you which construct-label pairs matter.
"""

import pandas as pd

res = pd.read_csv("rq2_construct_label_tests.csv")

# Sort all 56 by Cramer's V, largest first
res_sorted = res.sort_values("cramers_v", ascending=False)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)
print("--- All 56 pairs, sorted by Cramer's V ---")
print(res_sorted[["construct", "label", "family", "diff", "cramers_v", "p_holm", "sig_holm"]].to_string(index=False))

# Cutoff: small-to-moderate effect size threshold (conventional rule of thumb for 2x2 tables)
CRAMERS_V_CUTOFF = 0.10
top = res_sorted[res_sorted["cramers_v"] >= CRAMERS_V_CUTOFF]

print(f"\n--- Pairs with Cramer's V >= {CRAMERS_V_CUTOFF} ({len(top)} of 56) ---")
print(top[["construct", "label", "family", "diff", "cramers_v", "p_holm"]].to_string(index=False))

top.to_csv("rq2_top_effects.csv", index=False)
print(f"\nSaved: rq2_top_effects.csv ({len(top)} rows) — these are the candidates for cluster-bootstrap")

--- All 56 pairs, sorted by Cramer's V ---
construct                      label   family      diff  cramers_v        p_holm  sig_holm
       MD     Career Planning Advice    lloom  0.169479   0.175775  0.000000e+00      True
       EX          Emotional Support    lloom  0.124775   0.130628  0.000000e+00      True
       EX     Career Planning Advice    lloom  0.117009   0.128628  0.000000e+00      True
       EX          emotional_support taxonomy  0.115758   0.127798  0.000000e+00      True
      COG          emotional_support taxonomy  0.163662   0.123783  0.000000e+00      True
      COG          Emotional Support    lloom  0.162408   0.116481  0.000000e+00      True
      COG        Support Connections    lloom  0.140068   0.101476  0.000000e+00      True
      COG           Practical Advice    lloom  0.146956   0.101243  0.000000e+00      True
       MD          Emotional Support    lloom  0.101396   0.100147  0.000000e+00      True
       MD           tangible_support taxonomy -

### Cluster bootstrap for the contruct x label pairs 

In [14]:
"""
RQ2 — Cluster bootstrap for the construct x label pairs that clear Cohen's (1988)
small-effect threshold (Cramer's V >= 0.10 for 2x2 tables).

Why: comments cluster within posts (many comments share a parent post), so treating
each comment as an independent observation understates the true uncertainty. This
resamples POSTS with replacement (keeping all their comments together), recomputes
the weighted rate difference each time, and builds a percentile CI from that.

Uses rq2_merged_wide_bat.parquet (comment-level, already joined to BAT constructs).
"""

import pandas as pd
import numpy as np

merged = pd.read_parquet("rq2_merged_wide_bat.parquet")
res = pd.read_csv("rq2_construct_label_tests.csv")

# Cohen (1988) small-effect threshold for 2x2 tables (df* = 1): V = 0.10
CRAMERS_V_CUTOFF = 0.10
targets = res[res["cramers_v"] >= CRAMERS_V_CUTOFF].copy()
print(f"Bootstrapping {len(targets)} pairs (Cramer's V >= {CRAMERS_V_CUTOFF})")

N_BOOT = 2000
rng = np.random.default_rng(42)

unique_posts = merged["post_id"].unique()
n_posts = len(unique_posts)

# Pre-index comments by post for fast resampling
post_groups = merged.groupby("post_id").indices  # dict: post_id -> row indices
w_all = merged["w"].values if "w" in merged.columns else np.ones(len(merged))

def weighted_rate_diff(row_idx, construct_col, label_col):
    """Given a set of row indices (with repeats allowed), compute rate_in - rate_out."""
    c = merged[construct_col].values[row_idx].astype(bool)
    l = merged[label_col].values[row_idx].astype(bool)
    w = w_all[row_idx]

    w_in = w[c].sum()
    w_out = w[~c].sum()
    if w_in == 0 or w_out == 0:
        return np.nan

    rate_in = (w[c] * l[c]).sum() / w_in
    rate_out = (w[~c] * l[~c]).sum() / w_out
    return rate_in - rate_out

bootstrap_results = []

for _, row in targets.iterrows():
    construct = row["construct"]
    label = row["label"]
    observed_diff = row["diff"]

    boot_diffs = np.empty(N_BOOT)
    for b in range(N_BOOT):
        sampled_posts = rng.choice(unique_posts, size=n_posts, replace=True)
        # build row indices for this bootstrap sample (comments from resampled posts)
        idx = np.concatenate([post_groups[p] for p in sampled_posts])
        boot_diffs[b] = weighted_rate_diff(idx, construct, label)

    boot_diffs = boot_diffs[~np.isnan(boot_diffs)]
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    # bootstrap p-value: 2 * proportion of replicates on the other side of 0
    p_boot = 2 * min((boot_diffs > 0).mean(), (boot_diffs < 0).mean())
    p_boot = min(p_boot, 1.0)

    bootstrap_results.append({
        "construct": construct,
        "label": label,
        "observed_diff": observed_diff,
        "boot_ci_low": ci_low,
        "boot_ci_high": ci_high,
        "boot_mean": boot_diffs.mean(),
        "p_boot": p_boot,
        "ci_excludes_zero": (ci_low > 0) or (ci_high < 0),
    })
    print(f"{construct} / {label}: diff={observed_diff:.4f}, "
          f"CI=[{ci_low:.4f}, {ci_high:.4f}], p_boot={p_boot:.4f}")

boot_df = pd.DataFrame(bootstrap_results).sort_values("observed_diff", key=abs, ascending=False)
boot_df.to_csv("rq2_cluster_bootstrap_results.csv", index=False)

print("\n--- Summary ---")
print(boot_df.to_string(index=False))
n_robust = boot_df["ci_excludes_zero"].sum()
print(f"\n{n_robust} of {len(boot_df)} pairs have CI excluding zero after cluster bootstrap")
print("Saved: rq2_cluster_bootstrap_results.csv")

Bootstrapping 9 pairs (Cramer's V >= 0.1)
COG / emotional_support: diff=0.1637, CI=[0.1347, 0.1923], p_boot=0.0000
COG / Emotional Support: diff=0.1624, CI=[0.1328, 0.1910], p_boot=0.0000
COG / Support Connections: diff=0.1401, CI=[0.1148, 0.1656], p_boot=0.0000
COG / Practical Advice: diff=0.1470, CI=[0.1193, 0.1734], p_boot=0.0000
EX / Emotional Support: diff=0.1248, CI=[0.1050, 0.1446], p_boot=0.0000
EX / Career Planning Advice: diff=0.1170, CI=[0.0927, 0.1386], p_boot=0.0000
EX / emotional_support: diff=0.1158, CI=[0.0981, 0.1349], p_boot=0.0000
MD / Career Planning Advice: diff=0.1695, CI=[0.1452, 0.1926], p_boot=0.0000
MD / Emotional Support: diff=0.1014, CI=[0.0800, 0.1237], p_boot=0.0000

--- Summary ---
construct                  label  observed_diff  boot_ci_low  boot_ci_high  boot_mean  p_boot  ci_excludes_zero
       MD Career Planning Advice       0.169479     0.145189      0.192613   0.169316     0.0              True
      COG      emotional_support       0.163662     0.

### COG shows up 4 times in this shortlist and MD only twice — reinforcing what we flagged earlier: your empirical strongest pattern is actually around Cognitive Impairment, not Mental Distance, even though MD has been the theoretical focus (via the exit-intention link). Worth deciding how to frame that in the paper: either broaden the "mismatch" narrative to include COG as a co-lead finding, or keep MD as the throughline for theoretical reasons and treat COG's stronger empirical signal as a secondary/supporting finding.

### RQ3: Response type x exit-intention class 
 

In [16]:
"""
RQ3 — Step 1: Diagnostics before running exit-intention x response-type test
Goal: join score_sample_wide.parquet (comment-level) to exit_annotated_pass1_merged.csv
(post-level exit-intention classification) and check class distribution + match rate
before deciding on test design.
"""

import pandas as pd

wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")
exit_df = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/exit_annotated_pass1_merged.csv")  # adjust path if needed

print("exit_annotated columns:", list(exit_df.columns))
print("\nFirst few rows:")
print(exit_df.head())

# ---- Try to identify the key column and class column ----
key_candidates = [c for c in exit_df.columns if c.lower() in ("post_id", "id")]
print(f"\nPossible key columns: {key_candidates}")

# Print unique values for any column that looks categorical (few unique values)
print("\n--- Columns with <20 unique values (likely the class label) ---")
for c in exit_df.columns:
    n_unique = exit_df[c].nunique()
    if n_unique < 20:
        print(f"{c}: {n_unique} unique values -> {exit_df[c].unique()}")

# ---- Match rate check (once we know the key column) ----
# EDIT this after seeing the printed key_candidates above
exit_key_col = key_candidates[0] if key_candidates else None
if exit_key_col:
    wide_ids = set(wide["post_id"].unique())
    exit_ids = set(exit_df[exit_key_col].unique())
    match_rate = len(wide_ids & exit_ids) / len(wide_ids)
    print(f"\nUsing exit key column: {exit_key_col}")
    print(f"Match rate (wide post_id found in exit_df): {match_rate:.4f}")
    print(f"Wide unique post_ids: {len(wide_ids):,}")
    print(f"Exit_df unique post_ids: {len(exit_ids):,}")
    print(f"Unmatched wide post_ids: {len(wide_ids - exit_ids):,}")
else:
    print("\nCould not auto-detect key column — check columns list above manually.")

exit_annotated columns: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'exit_intention_class', 'exit_confidence', 'exit_evidence_span', 'exit_reasoning', 'exit_level', 'exit_reason_primary', 'exit_reason_secondary']

First few rows:
  row_type post_id  comment_id                                               text  triage  na_subtype  triage_reason  EX EMO COG  ...  \
0     post  cm8cbe         NaN  Armoring yourself with web presence DLP soluti...     NaN         NaN            NaN  NO  NO  NO  ...   
1     post  b88bq5         NaN  New CISO "To Do List"\nI will be starting as a...     NaN         NaN            NaN  NO  NO  NO  ...   
2     post  bpo012         NaN  Why CISOs are Suffering from Increasing Levels...     NaN         NaN            NaN  NO  NO  NO  ...   
3     post  8ln73a         NaN  Any advice for a new ISO?\nI just accepted

### 12,237 = posts with at least one BAT construct flagged (YES), out of the full 144,652-post corpus (bat_posts_results_final_patched.csv). This is "how many posts show burnout signal at all," regardless of whether anyone commented on them.

7,998 = unique posts that actually have comments in your comment sample (score_sample_wide.parquet). This is a different, smaller population for two reasons:

Silence — from your Phase 1 descriptives, ~71.6% of posts get zero comments at all. So even among BAT-positive posts, most never received a single reply. A post can be flagged MD/EX/whatever and still have nobody respond.
Sampling scope — your comment scoring sample was built as "full census of four small subreddits + 25,000 from r/sysadmin (reweighted)," which is itself a subset of all comments in the corpus, not comments on every single post.

In [17]:
"""
RQ3 — Step 2: exit_intention_class distribution within the 7,998 posts that have
comments in score_sample_wide.parquet. Confirms we have enough exit_explicit /
exit_contemplating posts to power the test.
"""

import pandas as pd

wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")
exit_df = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/exit_annotated_pass1_merged.csv")  # adjust path if needed

sample_post_ids = wide["post_id"].unique()
exit_sample = exit_df[exit_df["post_id"].isin(sample_post_ids)].drop_duplicates(subset="post_id")

print(f"Posts in comment sample: {len(sample_post_ids):,}")
print(f"Matched rows in exit_df (deduped by post_id): {len(exit_sample):,}")

print("\n--- exit_intention_class distribution (posts) ---")
print(exit_sample["exit_intention_class"].value_counts())
print(exit_sample["exit_intention_class"].value_counts(normalize=True))

# Merge onto comments to get the actual test n (comments, not posts)
merged = wide.merge(exit_sample[["post_id", "exit_intention_class", "exit_confidence"]],
                     on="post_id", how="left")

print("\n--- exit_intention_class distribution (comments — this is your test's n) ---")
print(merged["exit_intention_class"].value_counts())
print(merged["exit_intention_class"].value_counts(normalize=True))

print(f"\nComments with missing exit class: {merged['exit_intention_class'].isna().sum()}")

# Quick look at exit_confidence distribution — flag if low-confidence labels are common
print("\n--- exit_confidence summary ---")
print(merged["exit_confidence"].describe())
print(f"\nComments with exit_confidence < 0.7: {(merged['exit_confidence'] < 0.7).sum():,} "
      f"({(merged['exit_confidence'] < 0.7).mean():.4f})")

merged.to_parquet("rq3_merged_wide_exit.parquet")
print("\nSaved: rq3_merged_wide_exit.parquet")

Posts in comment sample: 7,998
Matched rows in exit_df (deduped by post_id): 7,998

--- exit_intention_class distribution (posts) ---
exit_intention_class
no_exit               5835
exit_contemplating    1699
exit_explicit          464
Name: count, dtype: int64
exit_intention_class
no_exit               0.729557
exit_contemplating    0.212428
exit_explicit         0.058015
Name: proportion, dtype: float64

--- exit_intention_class distribution (comments — this is your test's n) ---
exit_intention_class
no_exit               36391
exit_contemplating    13377
exit_explicit          3515
Name: count, dtype: int64
exit_intention_class
no_exit               0.682976
exit_contemplating    0.251056
exit_explicit         0.065969
Name: proportion, dtype: float64

Comments with missing exit class: 0

--- exit_confidence summary ---
count    53283.000000
mean         0.881674
std          0.192803
min          0.000000
25%          0.900000
50%          0.950000
75%          0.950000
max        

In [18]:
"""
RQ3 — Step 3: exit-intention class x response-type
For each of 14 response labels, build a 3x2 contingency table:
  rows = exit_intention_class (no_exit / exit_contemplating / exit_explicit)
  cols = label present / absent
Weighted by w. Cramer's V as effect size, standardized residuals per cell for
direction, chi-square p-value, Holm-corrected across all 14 tests.

Uses rq3_merged_wide_exit.parquet saved by step 2.
"""

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

merged = pd.read_parquet("rq3_merged_wide_exit.parquet")

EXIT_CLASSES = ["no_exit", "exit_contemplating", "exit_explicit"]

TAXONOMY_COLS = [
    "informational_support", "emotional_support", "esteem_support",
    "tangible_support", "network_support", "unsupportive_response",
]
LLOOM_COLS = [
    "Workplace Problem Guidance", "Career Planning Advice", "Emotional Support",
    "Critical Pushback", "Personal Relating", "Discussion Direction",
    "Practical Advice", "Support Connections",
]
ALL_LABELS = TAXONOMY_COLS + LLOOM_COLS

w = merged["w"] if "w" in merged.columns else pd.Series(1.0, index=merged.index)

overall_results = []
cell_results = []

for label in ALL_LABELS:
    l_flag = merged[label].astype(bool)

    # weighted 3x2 table: rows = exit class, cols = [label_yes, label_no]
    tab = np.zeros((3, 2))
    for i, cls in enumerate(EXIT_CLASSES):
        mask = merged["exit_intention_class"] == cls
        tab[i, 0] = w[mask & l_flag].sum()
        tab[i, 1] = w[mask & ~l_flag].sum()

    n = tab.sum()
    chi2, p, dof, expected = chi2_contingency(tab)
    # Cramer's V for a 3x2 table: k = min(rows, cols) = 2, so df* = 1 -> same 0.10/0.30/0.50 benchmarks
    k = min(tab.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * k))

    family = "taxonomy" if label in TAXONOMY_COLS else "lloom"

    overall_results.append({
        "label": label,
        "family": family,
        "chi2": chi2,
        "cramers_v": cramers_v,
        "p_raw": p,
        "n": n,
    })

    # per-class rate + standardized residual for the "label present" cell
    for i, cls in enumerate(EXIT_CLASSES):
        rate = tab[i, 0] / (tab[i, 0] + tab[i, 1])
        std_resid = (tab[i, 0] - expected[i, 0]) / np.sqrt(expected[i, 0])
        cell_results.append({
            "label": label,
            "family": family,
            "exit_class": cls,
            "rate": rate,
            "std_resid": std_resid,
        })

overall = pd.DataFrame(overall_results)
reject, p_holm, _, _ = multipletests(overall["p_raw"], method="holm")
overall["p_holm"] = p_holm
overall["sig_holm"] = reject
overall = overall.sort_values("cramers_v", ascending=False)

cells = pd.DataFrame(cell_results)
# pivot for easy reading: one row per label, one column per class's rate
pivot_rate = cells.pivot(index="label", columns="exit_class", values="rate")[EXIT_CLASSES]
pivot_resid = cells.pivot(index="label", columns="exit_class", values="std_resid")[EXIT_CLASSES]
pivot_resid.columns = [f"{c}_std_resid" for c in pivot_resid.columns]

full = overall.set_index("label").join(pivot_rate).join(pivot_resid).reset_index()
full = full.sort_values("cramers_v", ascending=False)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
print(full.to_string(index=False))

full.to_csv("rq3_exit_label_tests.csv", index=False)
print("\nSaved: rq3_exit_label_tests.csv")

# Cohen (1988) small-effect threshold, df*=1: V = 0.10
sig_and_meaningful = full[(full["sig_holm"]) & (full["cramers_v"] >= 0.10)]
print(f"\n{len(sig_and_meaningful)} of 14 labels significant AND >= Cohen's small threshold (0.10)")
print(sig_and_meaningful[["label", "family", "cramers_v", "p_holm"]].to_string(index=False))

                     label   family         chi2  cramers_v         p_raw        n        p_holm  sig_holm  no_exit  exit_contemplating  exit_explicit  no_exit_std_resid  exit_contemplating_std_resid  exit_explicit_std_resid
    Career Planning Advice    lloom 27833.350306   0.306417  0.000000e+00 296442.0  0.000000e+00      True 0.207893            0.529487       0.501528         -72.636093                    106.052759                56.366110
         Emotional Support    lloom  7450.064234   0.158530  0.000000e+00 296442.0  0.000000e+00      True 0.306031            0.471693       0.490149         -35.944106                     48.831447                33.949421
         emotional_support taxonomy  5861.213494   0.140612  0.000000e+00 296442.0  0.000000e+00      True 0.249572            0.386916       0.409059         -33.359726                     44.523896                32.830612
            esteem_support taxonomy  3662.487236   0.111152  0.000000e+00 296442.0  0.000000e+00    

### results: Emotional Support (LLooM): 30.6% → 47.2% → 49.0%
emotional_support (taxonomy): 25.0% → 38.7% → 40.9%
esteem_support: 22.7% → 32.0% → 37.0%
plausible story here: career-planning advice peaks mid-way (contemplating) and plateaus, while emotional/esteem support keeps rising all the way to the explicit-exit stage — a shift in what kind of support the community offers as intent solidifies. Worth checking tangible_support too, which actually declines monotonically (23.2% → 16.2% → 13.3%) even though it didn't clear the 0.10 cutoff — a smaller but directionally consistent signal that practical/logistical help drops off as exit intent increases.

### bootstrap

In [19]:
"""
RQ3 — Cluster bootstrap for the 4 labels that are both significant (Holm) and
>= Cohen's small threshold (Cramer's V >= 0.10): Career Planning Advice,
Emotional Support (lloom), emotional_support (taxonomy), esteem_support.

Resamples POSTS with replacement (not comments), recomputes Cramer's V and the
per-class rates each time, to get robust CIs accounting for within-post clustering.
"""

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

merged = pd.read_parquet("rq3_merged_wide_exit.parquet")

EXIT_CLASSES = ["no_exit", "exit_contemplating", "exit_explicit"]
TARGET_LABELS = ["Career Planning Advice", "Emotional Support", "emotional_support", "esteem_support"]

N_BOOT = 2000
rng = np.random.default_rng(42)

unique_posts = merged["post_id"].unique()
n_posts = len(unique_posts)
post_groups = merged.groupby("post_id").indices
w_all = merged["w"].values if "w" in merged.columns else np.ones(len(merged))
exit_class_all = merged["exit_intention_class"].values

def compute_stats(row_idx, label_col):
    l = merged[label_col].values[row_idx].astype(bool)
    w = w_all[row_idx]
    ec = exit_class_all[row_idx]

    tab = np.zeros((3, 2))
    rates = {}
    for i, cls in enumerate(EXIT_CLASSES):
        mask = ec == cls
        w_yes = w[mask & l].sum()
        w_no = w[mask & ~l].sum()
        tab[i, 0] = w_yes
        tab[i, 1] = w_no
        rates[cls] = w_yes / (w_yes + w_no) if (w_yes + w_no) > 0 else np.nan

    n = tab.sum()
    if (tab.sum(axis=1) == 0).any() or (tab.sum(axis=0) == 0).any():
        return np.nan, rates
    chi2, p, dof, expected = chi2_contingency(tab)
    cramers_v = np.sqrt(chi2 / (n * 1))  # k=1 for 3x2 table
    return cramers_v, rates

results = []
for label in TARGET_LABELS:
    boot_v = np.empty(N_BOOT)
    boot_rates = {cls: np.empty(N_BOOT) for cls in EXIT_CLASSES}

    for b in range(N_BOOT):
        sampled_posts = rng.choice(unique_posts, size=n_posts, replace=True)
        idx = np.concatenate([post_groups[p] for p in sampled_posts])
        v, rates = compute_stats(idx, label)
        boot_v[b] = v
        for cls in EXIT_CLASSES:
            boot_rates[cls][b] = rates[cls]

    boot_v_clean = boot_v[~np.isnan(boot_v)]
    v_ci = np.percentile(boot_v_clean, [2.5, 97.5])

    row = {
        "label": label,
        "cramers_v_boot_mean": boot_v_clean.mean(),
        "cramers_v_ci_low": v_ci[0],
        "cramers_v_ci_high": v_ci[1],
    }
    for cls in EXIT_CLASSES:
        r = boot_rates[cls][~np.isnan(boot_rates[cls])]
        ci = np.percentile(r, [2.5, 97.5])
        row[f"{cls}_rate_mean"] = r.mean()
        row[f"{cls}_rate_ci_low"] = ci[0]
        row[f"{cls}_rate_ci_high"] = ci[1]

    # contrast: contemplating vs explicit (tests the "plateau/dip" pattern)
    diff_ce = boot_rates["exit_explicit"] - boot_rates["exit_contemplating"]
    diff_ce = diff_ce[~np.isnan(diff_ce)]
    row["explicit_minus_contemplating_diff"] = diff_ce.mean()
    row["explicit_minus_contemplating_ci"] = tuple(np.percentile(diff_ce, [2.5, 97.5]))

    results.append(row)
    print(f"\n{label}")
    print(f"  Cramer's V: {row['cramers_v_boot_mean']:.4f}  CI=[{v_ci[0]:.4f}, {v_ci[1]:.4f}]")
    for cls in EXIT_CLASSES:
        print(f"  {cls}: {row[f'{cls}_rate_mean']:.4f}  CI=[{row[f'{cls}_rate_ci_low']:.4f}, {row[f'{cls}_rate_ci_high']:.4f}]")
    print(f"  explicit - contemplating diff: {row['explicit_minus_contemplating_diff']:.4f}  "
          f"CI={row['explicit_minus_contemplating_ci']}")

boot_df = pd.DataFrame(results)
boot_df.to_csv("rq3_cluster_bootstrap_results.csv", index=False)
print("\nSaved: rq3_cluster_bootstrap_results.csv")


Career Planning Advice
  Cramer's V: 0.3066  CI=[0.2835, 0.3312]
  no_exit: 0.2081  CI=[0.1971, 0.2190]
  exit_contemplating: 0.5299  CI=[0.5033, 0.5565]
  exit_explicit: 0.5012  CI=[0.4609, 0.5429]
  explicit - contemplating diff: -0.0287  CI=(-0.07835955392241696, 0.018362697423821163)

Emotional Support
  Cramer's V: 0.1587  CI=[0.1373, 0.1799]
  no_exit: 0.3061  CI=[0.2954, 0.3166]
  exit_contemplating: 0.4719  CI=[0.4499, 0.4957]
  exit_explicit: 0.4895  CI=[0.4542, 0.5243]
  explicit - contemplating diff: 0.0176  CI=(-0.024907209829214717, 0.05798583535329274)

emotional_support
  Cramer's V: 0.1412  CI=[0.1204, 0.1618]
  no_exit: 0.2495  CI=[0.2396, 0.2593]
  exit_contemplating: 0.3871  CI=[0.3650, 0.4110]
  exit_explicit: 0.4090  CI=[0.3746, 0.4443]
  explicit - contemplating diff: 0.0218  CI=(-0.018194217893921116, 0.06359030728959274)

esteem_support
  Cramer's V: 0.1117  CI=[0.0903, 0.1343]
  no_exit: 0.2270  CI=[0.2179, 0.2364]
  exit_contemplating: 0.3196  CI=[0.2995, 0.3

### Sub-question B — For posts that are BOTH a given construct AND have exit intent, does the response type look different than for posts with the construct but no exit intent?

In [21]:
"""
Sub-question B diagnostic: for each BAT construct, check comment counts in the
intersection of (construct present) x (exit_intention_class), to see if we have
enough power to test whether exit intention changes the response-type pattern
WITHIN a construct (interaction), not just each factor's main effect alone.

Needs both BAT flags and exit_intention_class on the same comment-level table —
building that join first since RQ2 and RQ3 used separate merged files.
"""

import pandas as pd

wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")
bat = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/bat_posts_results_final_patched.csv")
exit_df = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/exit_annotated_pass1_merged.csv")

CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]
for c in CONSTRUCT_COLS:
    bat[c] = bat[c].astype(str).str.strip().str.upper().eq("YES")

sample_post_ids = wide["post_id"].unique()
bat_sample = bat[bat["post_id"].isin(sample_post_ids)].drop_duplicates(subset="post_id")
exit_sample = exit_df[exit_df["post_id"].isin(sample_post_ids)].drop_duplicates(subset="post_id")

# Combined post-level table: constructs + exit class
post_level = bat_sample[["post_id"] + CONSTRUCT_COLS].merge(
    exit_sample[["post_id", "exit_intention_class", "exit_confidence"]],
    on="post_id", how="inner"
)
print(f"Posts with both BAT and exit data: {len(post_level):,} (should be 7,998)")

# Merge onto comments
merged = wide.merge(post_level, on="post_id", how="left")
print(f"Comments after merge: {len(merged):,}")
print(f"Comments with missing construct/exit data: {merged[CONSTRUCT_COLS + ['exit_intention_class']].isna().any(axis=1).sum()}")

merged.to_parquet("subq_b_merged_wide_bat_exit.parquet")
print("Saved: subq_b_merged_wide_bat_exit.parquet")

EXIT_CLASSES = ["no_exit", "exit_contemplating", "exit_explicit"]

print("\n=== Cell sizes: construct-present comments x exit class ===")
for construct in CONSTRUCT_COLS:
    print(f"\n--- {construct} present ---")
    sub = merged[merged[construct] == True]
    counts = sub["exit_intention_class"].value_counts()
    for cls in EXIT_CLASSES:
        n_posts = sub[sub["exit_intention_class"] == cls]["post_id"].nunique()
        n_comments = counts.get(cls, 0)
        print(f"  {cls}: {n_comments:,} comments across {n_posts:,} posts")

# Specifically flag the thinnest cells (construct present + exit_explicit)
print("\n=== Thinnest cells (construct present + exit_explicit) — the ones that matter most ===")
for construct in CONSTRUCT_COLS:
    sub = merged[(merged[construct] == True) & (merged["exit_intention_class"] == "exit_explicit")]
    n_posts = sub["post_id"].nunique()
    n_comments = len(sub)
    print(f"{construct} + exit_explicit: {n_comments:,} comments, {n_posts:,} posts")

Posts with both BAT and exit data: 7,998 (should be 7,998)
Comments after merge: 53,283
Comments with missing construct/exit data: 0
Saved: subq_b_merged_wide_bat_exit.parquet

=== Cell sizes: construct-present comments x exit class ===

--- EX present ---
  no_exit: 17,343 comments across 2,652 posts
  exit_contemplating: 8,044 comments across 1,063 posts
  exit_explicit: 2,469 comments across 305 posts

--- EMO present ---
  no_exit: 23,752 comments across 3,718 posts
  exit_contemplating: 9,002 comments across 1,084 posts
  exit_explicit: 2,714 comments across 335 posts

--- COG present ---
  no_exit: 6,002 comments across 1,095 posts
  exit_contemplating: 2,070 comments across 290 posts
  exit_explicit: 395 comments across 48 posts

--- MD present ---
  no_exit: 8,571 comments across 1,192 posts
  exit_contemplating: 8,755 comments across 958 posts
  exit_explicit: 2,623 comments across 299 posts

=== Thinnest cells (construct present + exit_explicit) — the ones that matter most ==

In [22]:
"""
Sub-question B: Does exit intention change the response-type pattern WITHIN a
given BAT construct? For each construct (EX, EMO, COG, MD), restrict to comments
where that construct is present, then run the same 3x2 (exit class x label)
test as RQ3, but on this restricted subset.

This tests the interaction, not just the main effects RQ2/RQ3 already captured.
COG is flagged as lower-power (thin exit_explicit cell, 48 posts).
"""

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

merged = pd.read_parquet("subq_b_merged_wide_bat_exit.parquet")

CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]
EXIT_CLASSES = ["no_exit", "exit_contemplating", "exit_explicit"]

TAXONOMY_COLS = [
    "informational_support", "emotional_support", "esteem_support",
    "tangible_support", "network_support", "unsupportive_response",
]
LLOOM_COLS = [
    "Workplace Problem Guidance", "Career Planning Advice", "Emotional Support",
    "Critical Pushback", "Personal Relating", "Discussion Direction",
    "Practical Advice", "Support Connections",
]
ALL_LABELS = TAXONOMY_COLS + LLOOM_COLS

all_rows = []

for construct in CONSTRUCT_COLS:
    sub = merged[merged[construct] == True].copy()
    w = sub["w"] if "w" in sub.columns else pd.Series(1.0, index=sub.index)

    construct_results = []
    for label in ALL_LABELS:
        l_flag = sub[label].astype(bool)

        tab = np.zeros((3, 2))
        for i, cls in enumerate(EXIT_CLASSES):
            mask = sub["exit_intention_class"] == cls
            tab[i, 0] = w[mask & l_flag].sum()
            tab[i, 1] = w[mask & ~l_flag].sum()

        n = tab.sum()
        if (tab.sum(axis=1) == 0).any() or (tab.sum(axis=0) == 0).any():
            continue

        chi2, p, dof, expected = chi2_contingency(tab)
        cramers_v = np.sqrt(chi2 / (n * 1))  # k=1 for 3x2

        rates = {cls: tab[i, 0] / (tab[i, 0] + tab[i, 1]) for i, cls in enumerate(EXIT_CLASSES)}
        family = "taxonomy" if label in TAXONOMY_COLS else "lloom"

        construct_results.append({
            "construct": construct,
            "label": label,
            "family": family,
            "chi2": chi2,
            "cramers_v": cramers_v,
            "p_raw": p,
            "n": n,
            "no_exit_rate": rates["no_exit"],
            "exit_contemplating_rate": rates["exit_contemplating"],
            "exit_explicit_rate": rates["exit_explicit"],
        })

    # Holm correction WITHIN each construct's 14 tests (separate families)
    cdf = pd.DataFrame(construct_results)
    reject, p_holm, _, _ = multipletests(cdf["p_raw"], method="holm")
    cdf["p_holm"] = p_holm
    cdf["sig_holm"] = reject
    all_rows.append(cdf)

full = pd.concat(all_rows, ignore_index=True)
full = full.sort_values(["construct", "cramers_v"], ascending=[True, False])

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
print(full[["construct", "label", "family", "cramers_v", "p_holm", "sig_holm",
            "no_exit_rate", "exit_contemplating_rate", "exit_explicit_rate", "n"]].to_string(index=False))

full.to_csv("subq_b_construct_exit_label_tests.csv", index=False)
print("\nSaved: subq_b_construct_exit_label_tests.csv")

# Cohen (1988) small threshold, df*=1: V >= 0.10
sig_and_meaningful = full[(full["sig_holm"]) & (full["cramers_v"] >= 0.10)]
print(f"\n{len(sig_and_meaningful)} of {len(full)} construct-label pairs significant AND >= Cohen's small threshold")
print(sig_and_meaningful[["construct", "label", "cramers_v", "p_holm"]].to_string(index=False))

print("\nNote: COG results use thin cells (48 posts in COG+exit_explicit) — treat as lower-confidence.")

construct                      label   family  cramers_v        p_holm  sig_holm  no_exit_rate  exit_contemplating_rate  exit_explicit_rate            n
      COG     Career Planning Advice    lloom   0.264282  0.000000e+00      True      0.283928                 0.558897            0.572195  40243.01812
      COG          Emotional Support    lloom   0.159534 5.076566e-222      True      0.441257                 0.625907            0.530365  40243.01812
      COG           tangible_support taxonomy   0.156139 1.087276e-212      True      0.341880                 0.210085            0.112352  40243.01812
      COG          emotional_support taxonomy   0.135974 2.973882e-161      True      0.389882                 0.545867            0.399890  40243.01812
      COG        Support Connections    lloom   0.103346  4.659426e-93      True      0.485332                 0.421632            0.273020  40243.01812
      COG             esteem_support taxonomy   0.090759  9.369308e-72      True  

## results: exit intention drives response type in largely the same way regardless of which BAT construct is flagged. Career Planning Advice, Emotional Support, and emotional_support show up as the top significant effects across all four constructs — not just MD

In [23]:
"""
Sub-question B — Cluster bootstrap for the 16 pairs that were significant AND
>= Cohen's small threshold. Resamples POSTS (within each construct's positive
subset) with replacement, recomputes Cramer's V each time.

Special attention to:
  - MD's pairs (are they really the smallest of the four, or does the bootstrap
    CI overlap with EX/EMO?)
  - COG's tangible_support / Support Connections (thin cells, 48 posts in
    COG+exit_explicit — check if these survive at all)
"""

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

merged = pd.read_parquet("subq_b_merged_wide_bat_exit.parquet")

EXIT_CLASSES = ["no_exit", "exit_contemplating", "exit_explicit"]

TARGET_PAIRS = [
    ("COG", "Career Planning Advice"), ("COG", "Emotional Support"),
    ("COG", "tangible_support"), ("COG", "emotional_support"),
    ("COG", "Support Connections"),
    ("EMO", "Career Planning Advice"), ("EMO", "Emotional Support"),
    ("EMO", "emotional_support"), ("EMO", "esteem_support"),
    ("EX", "Career Planning Advice"), ("EX", "Emotional Support"),
    ("EX", "emotional_support"), ("EX", "esteem_support"),
    ("MD", "Career Planning Advice"), ("MD", "Emotional Support"),
    ("MD", "emotional_support"),
]

N_BOOT = 2000
rng = np.random.default_rng(42)

def cramers_v_for_subset(sub_df, label):
    w = sub_df["w"] if "w" in sub_df.columns else pd.Series(1.0, index=sub_df.index)
    l = sub_df[label].astype(bool)
    tab = np.zeros((3, 2))
    for i, cls in enumerate(EXIT_CLASSES):
        mask = sub_df["exit_intention_class"] == cls
        tab[i, 0] = w[mask & l].sum()
        tab[i, 1] = w[mask & ~l].sum()
    n = tab.sum()
    if (tab.sum(axis=1) == 0).any() or (tab.sum(axis=0) == 0).any():
        return np.nan
    chi2, p, dof, expected = chi2_contingency(tab)
    return np.sqrt(chi2 / (n * 1))

results = []
for construct, label in TARGET_PAIRS:
    sub = merged[merged[construct] == True].copy()
    unique_posts = sub["post_id"].unique()
    n_posts = len(unique_posts)
    post_groups = sub.groupby("post_id").indices
    sub_reset = sub.reset_index(drop=True)
    post_groups_reset = sub_reset.groupby("post_id").indices

    observed_v = cramers_v_for_subset(sub_reset, label)

    boot_v = np.empty(N_BOOT)
    for b in range(N_BOOT):
        sampled_posts = rng.choice(unique_posts, size=n_posts, replace=True)
        idx = np.concatenate([post_groups_reset[p] for p in sampled_posts])
        boot_v[b] = cramers_v_for_subset(sub_reset.iloc[idx], label)

    boot_v_clean = boot_v[~np.isnan(boot_v)]
    ci = np.percentile(boot_v_clean, [2.5, 97.5])

    results.append({
        "construct": construct,
        "label": label,
        "observed_cramers_v": observed_v,
        "boot_mean_v": boot_v_clean.mean(),
        "boot_ci_low": ci[0],
        "boot_ci_high": ci[1],
        "n_posts_in_construct": n_posts,
    })
    print(f"{construct} / {label}: V={observed_v:.4f}  boot CI=[{ci[0]:.4f}, {ci[1]:.4f}]  (n_posts={n_posts})")

boot_df = pd.DataFrame(results).sort_values("observed_cramers_v", ascending=False)
boot_df.to_csv("subq_b_cluster_bootstrap_results.csv", index=False)

print("\n--- Summary, sorted by effect size ---")
print(boot_df.to_string(index=False))

print("\n--- Cross-construct comparison for shared labels ---")
for label in ["Career Planning Advice", "Emotional Support", "emotional_support"]:
    sub = boot_df[boot_df["label"] == label].sort_values("observed_cramers_v", ascending=False)
    print(f"\n{label}:")
    print(sub[["construct", "observed_cramers_v", "boot_ci_low", "boot_ci_high"]].to_string(index=False))

print("\nSaved: subq_b_cluster_bootstrap_results.csv")

COG / Career Planning Advice: V=0.2643  boot CI=[0.2168, 0.3144]  (n_posts=1433)
COG / Emotional Support: V=0.1595  boot CI=[0.1070, 0.2082]  (n_posts=1433)
COG / tangible_support: V=0.1561  boot CI=[0.1162, 0.1965]  (n_posts=1433)
COG / emotional_support: V=0.1360  boot CI=[0.0794, 0.1975]  (n_posts=1433)
COG / Support Connections: V=0.1033  boot CI=[0.0589, 0.1561]  (n_posts=1433)
EMO / Career Planning Advice: V=0.3151  boot CI=[0.2843, 0.3438]  (n_posts=5137)
EMO / Emotional Support: V=0.1839  boot CI=[0.1599, 0.2086]  (n_posts=5137)
EMO / emotional_support: V=0.1675  boot CI=[0.1439, 0.1947]  (n_posts=5137)
EMO / esteem_support: V=0.1284  boot CI=[0.1052, 0.1534]  (n_posts=5137)
EX / Career Planning Advice: V=0.3055  boot CI=[0.2738, 0.3351]  (n_posts=4020)
EX / Emotional Support: V=0.1714  boot CI=[0.1446, 0.2000]  (n_posts=4020)
EX / emotional_support: V=0.1488  boot CI=[0.1199, 0.1777]  (n_posts=4020)
EX / esteem_support: V=0.1310  boot CI=[0.1007, 0.1652]  (n_posts=4020)
MD / C

In [24]:
"""
Robustness check: does the construct ordering (EMO/EX > COG > MD) hold when we
restrict to "pure" posts — flagged for exactly ONE construct, no co-occurrence —
removing the confound where e.g. an MD+EX post's effect might really be driven
by its EX component.
"""

import pandas as pd

merged = pd.read_parquet("subq_b_merged_wide_bat_exit.parquet")
CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]
EXIT_CLASSES = ["no_exit", "exit_contemplating", "exit_explicit"]

# Post-level table (dedupe from comment-level merged)
post_level = merged.drop_duplicates(subset="post_id")[["post_id"] + CONSTRUCT_COLS + ["exit_intention_class"]]

n_flagged = post_level[CONSTRUCT_COLS].sum(axis=1)
print(f"Total posts: {len(post_level):,}")
print(f"Posts with exactly 1 construct flagged: {(n_flagged == 1).sum():,}")

print("\n--- 'Pure' single-construct post counts, by construct ---")
for c in CONSTRUCT_COLS:
    is_pure = (post_level[c] == True) & (n_flagged == 1)
    print(f"{c} pure: {is_pure.sum():,} posts")

print("\n--- 'Pure' single-construct posts x exit class (post-level) ---")
for c in CONSTRUCT_COLS:
    is_pure = (post_level[c] == True) & (n_flagged == 1)
    sub = post_level[is_pure]
    print(f"\n{c} pure (n={len(sub)}):")
    print(sub["exit_intention_class"].value_counts())

# Now comment-level counts (the actual test n)
print("\n--- 'Pure' single-construct comments x exit class ---")
pure_post_ids = {}
for c in CONSTRUCT_COLS:
    is_pure = (post_level[c] == True) & (n_flagged == 1)
    pure_post_ids[c] = set(post_level[is_pure]["post_id"])

for c in CONSTRUCT_COLS:
    sub_comments = merged[merged["post_id"].isin(pure_post_ids[c])]
    print(f"\n{c} pure: {len(sub_comments):,} comments across {sub_comments['post_id'].nunique():,} posts")
    counts = sub_comments["exit_intention_class"].value_counts()
    for cls in EXIT_CLASSES:
        n_posts = sub_comments[sub_comments["exit_intention_class"] == cls]["post_id"].nunique()
        print(f"  {cls}: {counts.get(cls, 0):,} comments, {n_posts:,} posts")

Total posts: 7,998
Posts with exactly 1 construct flagged: 4,595

--- 'Pure' single-construct post counts, by construct ---
EX pure: 1,295 posts
EMO pure: 2,238 posts
COG pure: 482 posts
MD pure: 580 posts

--- 'Pure' single-construct posts x exit class (post-level) ---

EX pure (n=1295):
exit_intention_class
no_exit               1026
exit_contemplating     219
exit_explicit           50
Name: count, dtype: int64

EMO pure (n=2238):
exit_intention_class
no_exit               1977
exit_contemplating     203
exit_explicit           58
Name: count, dtype: int64

COG pure (n=482):
exit_intention_class
no_exit               444
exit_contemplating     36
exit_explicit           2
Name: count, dtype: int64

MD pure (n=580):
exit_intention_class
no_exit               344
exit_contemplating    192
exit_explicit          44
Name: count, dtype: int64

--- 'Pure' single-construct comments x exit class ---

EX pure: 7,471 comments across 1,295 posts
  no_exit: 5,877 comments, 1,026 posts
  exit_co

In [25]:
"""
Robustness check: restrict to PURE single-construct posts (no co-occurring
construct) and test response-type x exit-intent (collapsed to no_exit vs
any_exit, since exit_explicit cells are too thin for a stable 3-class test
here). COG excluded — only 2 posts in exit_explicit, unusable.

Compares against the "any construct present" version from sub-question B to
see if the EMO/EX > MD ordering holds once co-occurrence is removed.
"""

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

merged = pd.read_parquet("subq_b_merged_wide_bat_exit.parquet")
CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]

post_level = merged.drop_duplicates(subset="post_id")[["post_id"] + CONSTRUCT_COLS]
n_flagged = post_level[CONSTRUCT_COLS].sum(axis=1)

pure_post_ids = {}
for c in ["EX", "EMO", "MD"]:  # COG excluded
    is_pure = (post_level[c] == True) & (n_flagged == 1)
    pure_post_ids[c] = set(post_level[is_pure]["post_id"])

merged["any_exit"] = merged["exit_intention_class"] != "no_exit"

TAXONOMY_COLS = [
    "informational_support", "emotional_support", "esteem_support",
    "tangible_support", "network_support", "unsupportive_response",
]
LLOOM_COLS = [
    "Workplace Problem Guidance", "Career Planning Advice", "Emotional Support",
    "Critical Pushback", "Personal Relating", "Discussion Direction",
    "Practical Advice", "Support Connections",
]
ALL_LABELS = TAXONOMY_COLS + LLOOM_COLS

all_rows = []
for construct in ["EX", "EMO", "MD"]:
    sub = merged[merged["post_id"].isin(pure_post_ids[construct])].copy()
    w = sub["w"] if "w" in sub.columns else pd.Series(1.0, index=sub.index)

    construct_results = []
    for label in ALL_LABELS:
        l_flag = sub[label].astype(bool)
        e_flag = sub["any_exit"]

        tab = np.zeros((2, 2))
        tab[0, 0] = w[e_flag & l_flag].sum()
        tab[0, 1] = w[e_flag & ~l_flag].sum()
        tab[1, 0] = w[~e_flag & l_flag].sum()
        tab[1, 1] = w[~e_flag & ~l_flag].sum()

        n = tab.sum()
        if (tab.sum(axis=1) == 0).any() or (tab.sum(axis=0) == 0).any():
            continue
        chi2, p, dof, expected = chi2_contingency(tab)
        cramers_v = np.sqrt(chi2 / n)  # 2x2 -> phi

        rate_exit = tab[0, 0] / (tab[0, 0] + tab[0, 1])
        rate_no_exit = tab[1, 0] / (tab[1, 0] + tab[1, 1])
        family = "taxonomy" if label in TAXONOMY_COLS else "lloom"

        construct_results.append({
            "construct": construct,
            "label": label,
            "family": family,
            "cramers_v": cramers_v,
            "p_raw": p,
            "rate_any_exit": rate_exit,
            "rate_no_exit": rate_no_exit,
            "diff": rate_exit - rate_no_exit,
            "n_posts_pure": len(pure_post_ids[construct]),
            "n_comments": n,
        })

    cdf = pd.DataFrame(construct_results)
    reject, p_holm, _, _ = multipletests(cdf["p_raw"], method="holm")
    cdf["p_holm"] = p_holm
    cdf["sig_holm"] = reject
    all_rows.append(cdf)

full = pd.concat(all_rows, ignore_index=True)
full = full.sort_values(["construct", "cramers_v"], ascending=[True, False])

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
print(full[["construct", "label", "family", "cramers_v", "p_holm", "sig_holm",
            "rate_no_exit", "rate_any_exit", "diff", "n_comments"]].to_string(index=False))

full.to_csv("pure_construct_exit_label_tests.csv", index=False)
print("\nSaved: pure_construct_exit_label_tests.csv")

print("\n--- Cross-construct comparison: Career Planning Advice (pure, 2-class) ---")
cpa = full[full["label"] == "Career Planning Advice"].sort_values("cramers_v", ascending=False)
print(cpa[["construct", "cramers_v", "diff", "n_comments"]].to_string(index=False))

print("\n--- Compare to sub-question B (any-construct-present) ordering for reference ---")
print("Sub-Q B: EMO=0.315, EX=0.305, COG=0.264, MD=0.215 (Career Planning Advice, 3-class)")

construct                      label   family  cramers_v        p_holm  sig_holm  rate_no_exit  rate_any_exit      diff  n_comments
      EMO     Career Planning Advice    lloom   0.297358  0.000000e+00      True      0.136751       0.488476  0.351725 92622.40136
      EMO          Emotional Support    lloom   0.123596 1.577813e-308      True      0.236384       0.403025  0.166641 92622.40136
      EMO          emotional_support taxonomy   0.109475 2.577245e-242      True      0.183125       0.318178  0.135053 92622.40136
      EMO             esteem_support taxonomy   0.085460 4.330660e-148      True      0.187442       0.293040  0.105599 92622.40136
      EMO           tangible_support taxonomy   0.065563  1.399781e-87      True      0.236440       0.151679 -0.084761 92622.40136
      EMO          Personal Relating    lloom   0.060324  2.524318e-74      True      0.557997       0.465232 -0.092765 92622.40136
      EMO Workplace Problem Guidance    lloom   0.050273  6.117015e-52      

### RQ4 — Subreddit stratum × response type
Does r/sysadmin respond differently than the four security-focused subreddits?

In [27]:
"""
RQ4 — Step 1: subreddit stratum distribution in the comment sample.
subreddit_source is already in score_sample_wide.parquet — no new join needed.
"""

import pandas as pd

wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")

SUBREDDITS = ["sysadmin", "cybersecurity", "SecurityCareerAdvice", "asknetsec", "ciso"]
 
TAXONOMY_COLS = [
    "informational_support", "emotional_support", "esteem_support",
    "tangible_support", "network_support", "unsupportive_response",
]
LLOOM_COLS = [
    "Workplace Problem Guidance", "Career Planning Advice", "Emotional Support",
    "Critical Pushback", "Personal Relating", "Discussion Direction",
    "Practical Advice", "Support Connections",
]
ALL_LABELS = TAXONOMY_COLS + LLOOM_COLS
 
overall_results = []
cell_results = []
 
for label in ALL_LABELS:
    l_flag = wide[label].astype(bool)
 
    tab = np.zeros((5, 2))
    for i, sub in enumerate(SUBREDDITS):
        mask = wide["subreddit_source"] == sub
        tab[i, 0] = (mask & l_flag).sum()
        tab[i, 1] = (mask & ~l_flag).sum()
 
    n = tab.sum()
    chi2, p, dof, expected = chi2_contingency(tab)
    k = min(tab.shape) - 1  # = 1
    cramers_v = np.sqrt(chi2 / (n * k))
 
    family = "taxonomy" if label in TAXONOMY_COLS else "lloom"
    overall_results.append({
        "label": label, "family": family, "chi2": chi2,
        "cramers_v": cramers_v, "p_raw": p, "n": n,
    })
 
    for i, sub in enumerate(SUBREDDITS):
        rate = tab[i, 0] / (tab[i, 0] + tab[i, 1])
        std_resid = (tab[i, 0] - expected[i, 0]) / np.sqrt(expected[i, 0])
        cell_results.append({
            "label": label, "subreddit": sub, "rate": rate, "std_resid": std_resid,
        })
 
overall = pd.DataFrame(overall_results)
reject, p_holm, _, _ = multipletests(overall["p_raw"], method="holm")
overall["p_holm"] = p_holm
overall["sig_holm"] = reject
overall = overall.sort_values("cramers_v", ascending=False)
 
cells = pd.DataFrame(cell_results)
pivot_rate = cells.pivot(index="label", columns="subreddit", values="rate")[SUBREDDITS]
pivot_resid = cells.pivot(index="label", columns="subreddit", values="std_resid")[SUBREDDITS]
pivot_resid.columns = [f"{s}_std_resid" for s in pivot_resid.columns]
 
full = overall.set_index("label").join(pivot_rate).join(pivot_resid).reset_index()
full = full.sort_values("cramers_v", ascending=False)
 
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)
print(full[["label", "family", "cramers_v", "p_holm", "sig_holm"] + SUBREDDITS].to_string(index=False))
 
full.to_csv("rq4_subreddit_label_tests.csv", index=False)
print("\nSaved: rq4_subreddit_label_tests.csv")
 
sig_and_meaningful = full[(full["sig_holm"]) & (full["cramers_v"] >= 0.10)]
print(f"\n{len(sig_and_meaningful)} of 14 labels significant AND >= Cohen's small threshold (0.10)")
print(sig_and_meaningful[["label", "family", "cramers_v", "p_holm"]].to_string(index=False))
 
print("\nNote: ciso (n=176 comments, 12 posts) is severely underpowered — treat its")
print("per-subreddit rates/residuals as descriptive only, not statistically reliable.")
 

                     label   family  cramers_v        p_holm  sig_holm  sysadmin  cybersecurity  SecurityCareerAdvice  asknetsec     ciso
    Career Planning Advice    lloom   0.271306  0.000000e+00      True   0.26844       0.497123              0.712631   0.497061 0.392045
Workplace Problem Guidance    lloom   0.149667 5.174552e-256      True   0.34952       0.227355              0.160695   0.241672 0.562500
     informational_support taxonomy   0.116544 3.058584e-154      True   0.72892       0.816445              0.853782   0.868060 0.852273
      Discussion Direction    lloom   0.113145 2.856770e-145      True   0.40076       0.500483              0.566051   0.538210 0.573864
         Personal Relating    lloom   0.080351  3.442361e-72      True   0.54496       0.489942              0.400651   0.418681 0.488636
          Practical Advice    lloom   0.074776  2.730700e-62      True   0.54824       0.588460              0.662685   0.709993 0.670455
         emotional_support taxonom

### RQ5 — Upvote score × response type
Which response types get upvoted more? Uses the score column already in your data — different flavor of test since score is continuous, not categorical (probably a group comparison or correlation rather than chi-square/Cramér's V).

In [28]:
"""
RQ5 — Step 1: score distribution diagnostics
score is continuous, so this needs a different test design than RQ2-RQ4
(no Cramer's V here). Checking skew/outliers first to decide between a
mean-based test (t-test/ANOVA) or a rank-based test (Mann-Whitney U), which
is more robust to skew and outliers.
"""

import pandas as pd
import numpy as np

wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")
master = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/master_comments_filtered.csv")  # has the score column per your schema notes

print("wide columns:", [c for c in wide.columns if "score" in c.lower() or c == "id"])
print("master columns:", [c for c in master.columns if "score" in c.lower() or c == "id"])

wide columns: ['id']
master columns: ['id', 'score']


/var/folders/41/b55bchyx62j34hmsgfhfk2gw0000gp/T/ipykernel_91618/2491159190.py:13: DtypeWarning: Columns (2,3,14) have mixed types. Specify dtype option on import or set low_memory=False.
  master = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/master_comments_filtered.csv")  # has the score column per your schema notes


In [30]:
"""
RQ5 — Step 2: score distribution diagnostics + id-dtype safety check before merge.
The earlier DtypeWarning on master_comments_filtered.csv means some columns have
mixed types on read — checking that 'id' itself isn't affected before joining.
"""

import pandas as pd
import numpy as np


wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")
master = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/master_comments_filtered.csv")  # has the score column per your schema notes


# ---- dtype check ----
print(f"wide['id'] dtype: {wide['id'].dtype}")
print(f"master['id'] dtype: {master['id'].dtype}")
print(f"master['score'] dtype: {master['score'].dtype}")

# force both to string for a safe join, avoid silent int/str mismatch
wide_ids = set(wide["id"].astype(str))
master_ids = set(master["id"].astype(str))
match_rate = len(wide_ids & master_ids) / len(wide_ids)
print(f"\nMatch rate (wide id found in master, as string): {match_rate:.4f}")
print(f"Unmatched: {len(wide_ids - master_ids):,}")

# ---- merge ----
wide["id_str"] = wide["id"].astype(str)
master["id_str"] = master["id"].astype(str)
merged = wide.merge(master[["id_str", "score"]], on="id_str", how="left")

print(f"\nComments with missing score after merge: {merged['score'].isna().sum():,}")

scores = merged["score"].dropna()

# ---- distribution diagnostics ----
print("\n--- Score distribution ---")
print(scores.describe())
print(f"\nSkewness: {scores.skew():.3f}")
print(f"Kurtosis: {scores.kurt():.3f}")
print(f"Negative scores: {(scores < 0).sum():,} ({(scores < 0).mean():.4f})")
print(f"Zero scores: {(scores == 0).sum():,} ({(scores == 0).mean():.4f})")
print(f"Score == 1 (default/no votes): {(scores == 1).sum():,} ({(scores == 1).mean():.4f})")

print("\n--- Percentiles ---")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{p}: {np.percentile(scores, p):.1f}")

print(f"\nMax score: {scores.max()}")
print(f"Top 10 scores:\n{scores.sort_values(ascending=False).head(10).values}")

merged.to_parquet("rq5_merged_wide_score.parquet")
print("\nSaved: rq5_merged_wide_score.parquet")

/var/folders/41/b55bchyx62j34hmsgfhfk2gw0000gp/T/ipykernel_91618/3694259318.py:12: DtypeWarning: Columns (2,3,14) have mixed types. Specify dtype option on import or set low_memory=False.
  master = pd.read_csv("/Users/nadia/Desktop/redditRun_june/comment_data/master_comments_filtered.csv")  # has the score column per your schema notes


wide['id'] dtype: object
master['id'] dtype: object
master['score'] dtype: float64

Match rate (wide id found in master, as string): 1.0000
Unmatched: 0

Comments with missing score after merge: 0

--- Score distribution ---
count    53283.000000
mean         8.547248
std         46.729727
min        -39.000000
25%          1.000000
50%          2.000000
75%          3.000000
max       2951.000000
Name: score, dtype: float64

Skewness: 22.201
Kurtosis: 784.077
Negative scores: 904 (0.0170)
Zero scores: 1,216 (0.0228)
Score == 1 (default/no votes): 23,180 (0.4350)

--- Percentiles ---
  p1: -2.0
  p5: 1.0
  p10: 1.0
  p25: 1.0
  p50: 2.0
  p75: 3.0
  p90: 12.0
  p95: 28.0
  p99: 137.0

Max score: 2951.0
Top 10 scores:
[2951. 2106. 1973. 1775. 1769. 1724. 1667. 1541. 1534. 1444.]

Saved: rq5_merged_wide_score.parquet


In [31]:
"""
RQ5 — Two tests per label:
(A) Engagement: does having this label change the odds a comment gets ANY vote
    at all (score != 1, i.e. someone besides the auto-1 interacted with it)?
    Weighted 2x2 chi-square / Cramer's V, same framework as RQ2/RQ3.
(B) Upvote rank: among comments that DID get engagement, do labeled comments
    rank higher or lower in score than unlabeled ones? Mann-Whitney U
    (robust to the extreme skew/outliers found in diagnostics), with
    rank-biserial correlation as effect size (-1 to 1, analogous magnitude
    read to Cramer's V: ~0.1 small, ~0.3 medium, ~0.5 large).

Uses rq5_merged_wide_score.parquet from step 2.
"""

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, mannwhitneyu
from statsmodels.stats.multitest import multipletests

merged = pd.read_parquet("rq5_merged_wide_score.parquet")

TAXONOMY_COLS = [
    "informational_support", "emotional_support", "esteem_support",
    "tangible_support", "network_support", "unsupportive_response",
]
LLOOM_COLS = [
    "Workplace Problem Guidance", "Career Planning Advice", "Emotional Support",
    "Critical Pushback", "Personal Relating", "Discussion Direction",
    "Practical Advice", "Support Connections",
]
ALL_LABELS = TAXONOMY_COLS + LLOOM_COLS

merged["any_engagement"] = merged["score"] != 1
w = merged["w"] if "w" in merged.columns else pd.Series(1.0, index=merged.index)

# ---------------- Test A: engagement (weighted 2x2) ----------------
engagement_results = []
for label in ALL_LABELS:
    l_flag = merged[label].astype(bool)
    e_flag = merged["any_engagement"]

    tab = np.zeros((2, 2))
    tab[0, 0] = w[l_flag & e_flag].sum()
    tab[0, 1] = w[l_flag & ~e_flag].sum()
    tab[1, 0] = w[~l_flag & e_flag].sum()
    tab[1, 1] = w[~l_flag & ~e_flag].sum()

    n = tab.sum()
    chi2, p, dof, expected = chi2_contingency(tab)
    cramers_v = np.sqrt(chi2 / n)

    rate_labeled = tab[0, 0] / (tab[0, 0] + tab[0, 1])
    rate_unlabeled = tab[1, 0] / (tab[1, 0] + tab[1, 1])
    family = "taxonomy" if label in TAXONOMY_COLS else "lloom"

    engagement_results.append({
        "label": label, "family": family, "cramers_v": cramers_v, "p_raw": p,
        "engagement_rate_labeled": rate_labeled,
        "engagement_rate_unlabeled": rate_unlabeled,
        "diff": rate_labeled - rate_unlabeled,
    })

eng_df = pd.DataFrame(engagement_results)
reject, p_holm, _, _ = multipletests(eng_df["p_raw"], method="holm")
eng_df["p_holm"] = p_holm
eng_df["sig_holm"] = reject
eng_df = eng_df.sort_values("cramers_v", ascending=False)

# ---------------- Test B: upvote rank among engaged comments (unweighted) ----------------
engaged = merged[merged["any_engagement"]].copy()

rank_results = []
for label in ALL_LABELS:
    l_flag = engaged[label].astype(bool)
    scores_labeled = engaged.loc[l_flag, "score"]
    scores_unlabeled = engaged.loc[~l_flag, "score"]

    if len(scores_labeled) < 5 or len(scores_unlabeled) < 5:
        continue

    stat, p = mannwhitneyu(scores_labeled, scores_unlabeled, alternative="two-sided")
    n1, n2 = len(scores_labeled), len(scores_unlabeled)
    # rank-biserial correlation effect size
    rank_biserial = 1 - (2 * stat) / (n1 * n2)

    family = "taxonomy" if label in TAXONOMY_COLS else "lloom"
    rank_results.append({
        "label": label, "family": family,
        "median_score_labeled": scores_labeled.median(),
        "median_score_unlabeled": scores_unlabeled.median(),
        "mean_score_labeled": scores_labeled.mean(),
        "mean_score_unlabeled": scores_unlabeled.mean(),
        "rank_biserial": rank_biserial,
        "p_raw": p,
        "n_labeled": n1, "n_unlabeled": n2,
    })

rank_df = pd.DataFrame(rank_results)
reject, p_holm, _, _ = multipletests(rank_df["p_raw"], method="holm")
rank_df["p_holm"] = p_holm
rank_df["sig_holm"] = reject
rank_df = rank_df.sort_values("rank_biserial", key=abs, ascending=False)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("=== Test A: Engagement (any vote, weighted) ===")
print(eng_df[["label", "family", "cramers_v", "p_holm", "sig_holm",
              "engagement_rate_labeled", "engagement_rate_unlabeled", "diff"]].to_string(index=False))

print("\n=== Test B: Upvote rank among engaged comments (unweighted, Mann-Whitney) ===")
print(rank_df[["label", "family", "rank_biserial", "p_holm", "sig_holm",
               "median_score_labeled", "median_score_unlabeled",
               "mean_score_labeled", "mean_score_unlabeled"]].to_string(index=False))

eng_df.to_csv("rq5_engagement_tests.csv", index=False)
rank_df.to_csv("rq5_rank_tests.csv", index=False)
print("\nSaved: rq5_engagement_tests.csv, rq5_rank_tests.csv")

print("\n--- Significant AND meaningful (Cohen's 0.10 for V; using same 0.10 rough guide for rank-biserial) ---")
print("\nEngagement:")
print(eng_df[(eng_df["sig_holm"]) & (eng_df["cramers_v"] >= 0.10)][["label", "cramers_v", "diff"]].to_string(index=False))
print("\nUpvote rank:")
print(rank_df[(rank_df["sig_holm"]) & (rank_df["rank_biserial"].abs() >= 0.10)][["label", "rank_biserial"]].to_string(index=False))

=== Test A: Engagement (any vote, weighted) ===
                     label   family  cramers_v        p_holm  sig_holm  engagement_rate_labeled  engagement_rate_unlabeled      diff
         Critical Pushback    lloom   0.062927 4.082669e-256      True                 0.581175                   0.518080  0.063095
     unsupportive_response taxonomy   0.046237 9.928723e-139      True                 0.592517                   0.534576  0.057941
     informational_support taxonomy   0.045052 8.674470e-132      True                 0.559334                   0.508320  0.051013
      Discussion Direction    lloom   0.039205 4.686359e-100      True                 0.569329                   0.529651  0.039678
         Personal Relating    lloom   0.034902  1.620377e-79      True                 0.529874                   0.564738 -0.034864
Workplace Problem Guidance    lloom   0.027510  9.202938e-50      True                 0.565158                   0.536185  0.028973
    Career Planning A

## RQ6 — Concept co-occurrence matrix
Which LLooM concepts tend to appear together in the same comment.

In [32]:
"""
RQ6 — Which LLooM-induced concepts tend to co-occur in the same comment?
Descriptive, not a hypothesis test. Three complementary views:
  1. Phi coefficient matrix (weighted) — binary correlation, -1 to 1
  2. Raw co-occurrence counts (unweighted) — how many comments have both
  3. Jaccard similarity — overlap size relative to union, ignores how common
     each label is individually (useful since labels have very different
     base rates, e.g. Practical Advice 57.7% vs unsupportive_response 19.2%)
"""

import pandas as pd
import numpy as np

wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")

LLOOM_COLS = [
    "Workplace Problem Guidance", "Career Planning Advice", "Emotional Support",
    "Critical Pushback", "Personal Relating", "Discussion Direction",
    "Practical Advice", "Support Connections",
]

w = wide["w"] if "w" in wide.columns else pd.Series(1.0, index=wide.index)

n_concepts = len(LLOOM_COLS)

# ---- 1. Weighted phi coefficient matrix ----
phi_matrix = pd.DataFrame(index=LLOOM_COLS, columns=LLOOM_COLS, dtype=float)
for i, c1 in enumerate(LLOOM_COLS):
    for j, c2 in enumerate(LLOOM_COLS):
        a = wide[c1].astype(bool)
        b = wide[c2].astype(bool)
        n11 = w[a & b].sum()
        n10 = w[a & ~b].sum()
        n01 = w[~a & b].sum()
        n00 = w[~a & ~b].sum()
        n = n11 + n10 + n01 + n00
        denom = np.sqrt((n11+n10)*(n01+n00)*(n11+n01)*(n10+n00))
        phi = (n11*n00 - n10*n01) / denom if denom > 0 else np.nan
        phi_matrix.loc[c1, c2] = phi

# ---- 2. Raw co-occurrence counts (unweighted) ----
cooc_counts = pd.DataFrame(index=LLOOM_COLS, columns=LLOOM_COLS, dtype=int)
for c1 in LLOOM_COLS:
    for c2 in LLOOM_COLS:
        cooc_counts.loc[c1, c2] = (wide[c1].astype(bool) & wide[c2].astype(bool)).sum()

# ---- 3. Jaccard similarity ----
jaccard_matrix = pd.DataFrame(index=LLOOM_COLS, columns=LLOOM_COLS, dtype=float)
for c1 in LLOOM_COLS:
    for c2 in LLOOM_COLS:
        a = wide[c1].astype(bool)
        b = wide[c2].astype(bool)
        union = (a | b).sum()
        intersect = (a & b).sum()
        jaccard_matrix.loc[c1, c2] = intersect / union if union > 0 else np.nan

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.3f}".format)

print("=== Weighted phi coefficient matrix ===")
print(phi_matrix)

print("\n=== Raw co-occurrence counts (unweighted, n comments with both) ===")
print(cooc_counts)

print("\n=== Jaccard similarity matrix ===")
print(jaccard_matrix)

# ---- Ranked pair list (off-diagonal, deduplicated) ----
pairs = []
for i, c1 in enumerate(LLOOM_COLS):
    for j, c2 in enumerate(LLOOM_COLS):
        if i < j:
            pairs.append({
                "concept_1": c1, "concept_2": c2,
                "phi": phi_matrix.loc[c1, c2],
                "jaccard": jaccard_matrix.loc[c1, c2],
                "n_cooccur": cooc_counts.loc[c1, c2],
            })
pairs_df = pd.DataFrame(pairs).sort_values("phi", ascending=False)
print("\n=== All 28 pairs, ranked by phi coefficient ===")
print(pairs_df.to_string(index=False))

phi_matrix.to_csv("rq6_phi_matrix.csv")
jaccard_matrix.to_csv("rq6_jaccard_matrix.csv")
cooc_counts.to_csv("rq6_cooccurrence_counts.csv")
pairs_df.to_csv("rq6_ranked_pairs.csv", index=False)
print("\nSaved: rq6_phi_matrix.csv, rq6_jaccard_matrix.csv, rq6_cooccurrence_counts.csv, rq6_ranked_pairs.csv")

=== Weighted phi coefficient matrix ===
                            Workplace Problem Guidance  Career Planning Advice  Emotional Support  Critical Pushback  Personal Relating  Discussion Direction  Practical Advice  Support Connections
Workplace Problem Guidance                       1.000                   0.166              0.180              0.143             -0.129                 0.284             0.519                0.297
Career Planning Advice                           0.166                   1.000              0.295              0.106             -0.086                 0.125             0.314                0.132
Emotional Support                                0.180                   0.295              1.000             -0.040              0.062                 0.017             0.136                0.030
Critical Pushback                                0.143                   0.106             -0.040              1.000             -0.275                 0.366             0.